In [1]:
import os
print(os.listdir('.'))
# Hoặc tìm kiếm file penguins_size.csv trên Colab

['.config', 'penguins_size.csv', 'sample_data']


In [2]:
#Import thư viện & Khai báo hằng số
import os
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

RANDOM_STATE = 42
DATA_PATH = "penguins_size.csv"
SCHEMA_PATH = "schema.json"

In [3]:
#Đọc dữ liệu & Kiểm tra ban đầu
# Đọc file dữ liệu
df = pd.read_csv(DATA_PATH)

print("Số lượng dòng trùng lặp:", df.duplicated().sum())
print("Các giá trị duy nhất trong cột sex ban đầu:", df["sex"].unique())
print("Tổng số dòng ban đầu:", len(df))

Số lượng dòng trùng lặp: 0
Các giá trị duy nhất trong cột sex ban đầu: ['MALE' 'FEMALE' nan '.']
Tổng số dòng ban đầu: 344


In [4]:
#Làm sạch dữ liệu
# 1. Chuẩn hoá giá trị lỗi "." trong cột sex thành NaN
df["sex"] = df["sex"].replace(".", np.nan)

# 2. Loại các dòng thiếu TOÀN BỘ 4 cột số đo
num_cols = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
df = df.dropna(subset=num_cols, how="all").reset_index(drop=True)

# 3. Loại các dòng thiếu nhãn target (nếu có)
df = df.dropna(subset=["species"]).reset_index(drop=True)

cat_cols = ["island", "sex"]
target_col = "species"

print("Số lượng dòng sau khi làm sạch:", len(df))
print("Các giá trị trong cột sex sau làm sạch:", df["sex"].unique())

Số lượng dòng sau khi làm sạch: 342
Các giá trị trong cột sex sau làm sạch: ['MALE' 'FEMALE' nan]


In [5]:
#Tách đặc trưng, Mã hóa nhãn & Chia Train/Test (Tránh Data Leakage)
X = df[num_cols + cat_cols]
y = df[target_col]

# Mã hoá nhãn target (Adelie=0, Chinstrap=1, Gentoo=2)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Mapping nhãn target:", dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))

# Chia dữ liệu TRƯỚC khi fit scaler/imputer
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
)

print(f"Kích thước X_train: {X_train.shape}")
print(f"Kích thước X_test:  {X_test.shape}")

Mapping nhãn target: {'Adelie': 0, 'Chinstrap': 1, 'Gentoo': 2}
Kích thước X_train: (273, 6)
Kích thước X_test:  (69, 6)


In [6]:
#Xây dựng Preprocessing Pipeline với ColumnTransformer
# Pipeline xử lý cột số
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Pipeline xử lý cột phân loại
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# Gom thành ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols),
])

# Chạy thử transform trên X_train để kiểm tra
X_train_transformed = preprocessor.fit_transform(X_train)
print("Kích thước dữ liệu sau khi biến đổi:", X_train_transformed.shape)

Kích thước dữ liệu sau khi biến đổi: (273, 9)


In [7]:
#Trích xuất và Xuất file schema.json (Cell cuối cùng)
# Tạo thư mục chứa schema nếu chưa có
schema = {
    "target": "species",
    "labels": list(label_encoder.classes_),
    "features": {
        "culmen_length_mm": {
            "type": "float",
            "min": round(float(df["culmen_length_mm"].min()), 1),
            "max": round(float(df["culmen_length_mm"].max()), 1)
        },
        "culmen_depth_mm": {
            "type": "float",
            "min": round(float(df["culmen_depth_mm"].min()), 1),
            "max": round(float(df["culmen_depth_mm"].max()), 1)
        },
        "flipper_length_mm": {
            "type": "float",
            "min": round(float(df["flipper_length_mm"].min()), 1),
            "max": round(float(df["flipper_length_mm"].max()), 1)
        },
        "body_mass_g": {
            "type": "float",
            "min": round(float(df["body_mass_g"].min()), 1),
            "max": round(float(df["body_mass_g"].max()), 1)
        },
        "island": {
            "type": "category",
            "values": sorted(df["island"].dropna().unique().tolist())
        },
        "sex": {
            "type": "category",
            "values": [s for s in df["sex"].dropna().unique().tolist() if s != "."]
        },
    }
}

with open(SCHEMA_PATH, "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

print(f"Đã lưu schema thành công tại: {SCHEMA_PATH}")

Đã lưu schema thành công tại: schema.json
